# ECE1512 Project A - Build an ABMIL training pipeline in PyTorch

This notebook uses pre-extracted whole-slide histopathology features and guides you through a complete PyTorch classification pipeline. The goal is not merely to run ABMIL; it is to understand the code path from data loading to optimization, validation, checkpoint selection, and final testing.

Please complete every cell marked `TODO`. Dataset engineering and metric boilerplate are provided so you can focus on the machine-learning training workflow.

This project requires basic familiarity with Python object-oriented programming (OOP). If you have limited experience with Python OOP, please review the provided introductory video before starting the project. 

[PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)

[Python Object-Oriented Programming ](https://www.youtube.com/watch?v=JeznW_7DlB0)

**Research extension:** Propose two research ideas and implement and evaluate one selected idea. Report ABMIL baseline and selected-extension results on **CAMELYON16, CAMELYON17, and BRACS**, using the same features and train/validation/test splits within each dataset. 


## Highlighted completed work

<mark style="background-color: #fff2a8; color: #1f2328;">Yellow highlights identify the filled code and written answers.</mark>

Each code callout lists the original TODO identifier, the line number in the executable cell immediately below it, and the completed statement. The runnable code follows each callout. The research-extension callouts identify the added implementation cells.

| Completed code | Written answers |
|---|---|
| [Step 1: DataLoaders (1A-1B)](#completed-code-step-1) | [Dataset and DataLoader answers](#completed-answers-data) |
| [Step 2: ABMIL model (2A-2E, 2G-2I)](#completed-code-step-2) | [Model and attention answers](#completed-answers-model) |
| [Step 3: Loss and optimizer (3A-3B)](#completed-code-step-3) | [Training and evaluation answers](#completed-answers-training) |
| [Step 4: Training epoch (4A-4G)](#completed-code-step-4) | [Two research proposals and protocol](#completed-research-answers) |
| [Step 5: Evaluation (5A-5D)](#completed-code-step-5) | |
| [Step 9: Research extension](#completed-extension-settings) | |

Search for **Completed code** or **Completed written answers** to locate the highlighted sections.


## Learning objectives

After this project you should be able to explain and implement:

1. `Dataset` vs. `DataLoader`, batching, shuffling.
2. `nn.Module`, trainable parameters, and `forward()`.
3. A classification loss and optimizer.
4. The training sequence `zero_grad -> forward -> loss -> backward -> step`.
5. The difference between `model.train()` and `model.eval()`.

## Step 0. Download Pre-extracted WSI Features & Select the Project Directory
Download the required feature file(s) from Zenodo or Hugging Face:

https://zenodo.org/records/23130543

https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature


Place the HDF5 file under the path expected by your YAML configuration in the `config/`, or edit data_dir in the YAML file.


In [ ]:
# Local: select .venv/Scripts/python.exe as the notebook kernel.
# If setting up another environment, uncomment these lines and restart the kernel:
# %pip install -r requirements.txt
# %pip install ipykernel
import importlib.util
required = ['torch', 'torchmetrics', 'h5py', 'pandas', 'sklearn', 'yaml', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Install requirements.txt in this notebook's kernel.")
print('Training dependencies are available.')

In [ ]:
import os, sys, copy, time, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from IPython.display import display, Markdown

# Small MIL batches run better without excessive CPU thread overhead.
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

If you are using Google Colab with Drive, mount Drive first and change `PROJECT_DIR`.

In [ ]:
# Optional in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
PROJECT_DIR = Path('./').resolve()  # Set to the project folder if using Colab.
DATASET = 'C16'  # Choose 'C16', 'C17', or 'BRACS'.
CONFIG_FILES = {
    'C16': 'camelyon16_medical_ssl_config.yml',
    'C17': 'camelyon17_medical_ssl_config.yml',
    'BRACS': 'bracs_medical_ssl_config.yml',
}
CONFIG_PATH = PROJECT_DIR / 'config' / CONFIG_FILES[DATASET]
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)  # YAML feature/metadata paths are relative to the project.
OUTPUT_DIR = PROJECT_DIR / 'results'
OUTPUT_DIR.mkdir(exist_ok=True)
print('Project:', PROJECT_DIR)
print('Config:', CONFIG_PATH)
from wsi_visualization import visualize_wsi_attention

In [ ]:
from dataloader import build_HDF5_feat_dataset, mil_collate_fn
from utils import load_config, seed_everything, infer_feature_dim, classification_metrics

conf = load_config(CONFIG_PATH)
seed_everything(int(conf.seed))
print(conf)
if not Path(conf.data_dir).is_file():
    raise FileNotFoundError(f'Update data_dir in {CONFIG_PATH}: {conf.data_dir}')
print('Feature file:', Path(conf.data_dir).resolve())

# Step 1 - Dataset and DataLoader

`build_HDF5_feat_dataset` is provided. It creates train/validation/test `Dataset` objects without loading all slide features into memory at once.


Complete the three `DataLoader` definitions. Think about which split should be shuffled.

[Download WSI datasets](https://zenodo.org/records/23130543)

[Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)


[PyTorch DataLoader](https://docs.pytorch.org/docs/2.14/data.html#torch.utils.data.DataLoader)


[PyTorch Datasets](https://docs.pytorch.org/vision/stable/datasets.html)


In [ ]:
train_data, val_data, test_data = build_HDF5_feat_dataset(conf.data_dir, conf)
conf.D_feat = infer_feature_dim(train_data)
print('Split sizes: Train {}, Validation {}, Test {};'.format(len(train_data), len(val_data), len(test_data)))
print("Feature Shape: {};".format(tuple(train_data[1]['input'].shape)))
print("Label: {}; \nSlide ID: {}.".format(train_data[1]['label'], train_data[1]['slide_id']))

In [ ]:
batch_size=conf.B
num_workers=int(conf.n_worker)
pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
collate_fn=mil_collate_fn
print("DataLoader args: Batch size {}; \n \
      # CPUs {};  \n \
      pin_memory {};  \n \
      collate_fn {}".format(batch_size, num_workers, pin_memory, collate_fn))


<a id="completed-code-step-1"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Step 1: DataLoaders

Filled statements in the executable cell below:

<pre><code>1A | line 07 | <mark style="background-color: #fff2a8; color: #1f2328;">train_loader = DataLoader(train_data, shuffle=True, generator=generator, **common)</mark>
1B | line 08 | <mark style="background-color: #fff2a8; color: #1f2328;">val_loader = DataLoader(val_data, shuffle=False, **common)</mark>
1B | line 09 | <mark style="background-color: #fff2a8; color: #1f2328;">test_loader = DataLoader(test_data, shuffle=False, **common)</mark></code></pre>

In [ ]:
def make_loaders(train_data, val_data, test_data, conf):
    common = dict(batch_size=int(conf.B), num_workers=int(conf.n_worker),
                  pin_memory=bool(conf.pin_memory) and torch.cuda.is_available(),
                  collate_fn=mil_collate_fn)
    # A separate generator keeps the training order reproducible across ablations.
    generator = torch.Generator().manual_seed(int(conf.seed))
    train_loader = DataLoader(train_data, shuffle=True, generator=generator, **common)
    val_loader = DataLoader(val_data, shuffle=False, **common)
    test_loader = DataLoader(test_data, shuffle=False, **common)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(train_data, val_data, test_data, conf)

In [ ]:
# Inspect one MIL batch.
batch = next(iter(train_loader))
print('input:', tuple(batch['input'].shape), batch['input'].dtype)
print('label:', tuple(batch['label'].shape), batch['label'].dtype)
print('slide IDs:', batch['slide_id'])

<a id="completed-answers-data"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed written answers</mark> — Dataset and DataLoader answers

- <mark style="background-color: #fff2a8; color: #1f2328;">A <code>Dataset</code> maps a slide index to its features, label, coordinates, and slide ID. A <code>DataLoader</code> selects indices, shuffles when requested, reads samples, and collates them into batches.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">Only the training loader uses <code>shuffle=True</code>, to vary gradient-update order. Validation and test loaders keep a fixed order for reproducible evaluation and prediction export.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">In <code>[B, N, D]</code>, <code>B</code> is the number of slides, <code>N</code> is the largest patch count in that batch after padding, and <code>D</code> is the extracted feature dimension. <code>mask[b, n]</code> is true only for real patches; padding must receive zero attention.</mark>

# Step 2 - Implement the ABMIL model

For each patch feature $h_i$, the gated attention score follows Ilse et al. (2018):

$$
a_i = w^T\left[\tanh(Vh_i) \odot \sigma(Uh_i)\right].
$$

The attention logits are normalized over instances:

$$
\alpha_i = \frac{\exp(a_i)}{\sum_j \exp(a_j)},
$$

and the slide representation is

$$
z = \sum_i \alpha_i h_i.
$$

You will implement the learnable layers and the key `forward()` operations.

[Custom nn Modules](https://docs.pytorch.org/tutorials/beginner/examples_nn/polynomial_module.html)

[nn.Linear](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html)

[nn.modules.dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html)

[nn.ReLU](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html)

[torch.nn.functional.softmax (F.softmax)](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.softmax.html)

[torch.bmm](https://docs.pytorch.org/docs/2.14/generated/torch.bmm.html)

<a id="completed-code-step-2"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Step 2: ABMIL model

Filled statements in the executable cell below:

<pre><code>2A | line 05 | <mark style="background-color: #fff2a8; color: #1f2328;">self.fc = nn.Linear(n_channels, n_classes)</mark>
2B | line 08 | <mark style="background-color: #fff2a8; color: #1f2328;">return self.fc(self.dropout(x))</mark>
2C | line 14 | <mark style="background-color: #fff2a8; color: #1f2328;">self.fc = nn.Linear(n_channels, m_dim)</mark>
2C | line 15 | <mark style="background-color: #fff2a8; color: #1f2328;">self.relu = nn.ReLU()</mark>
2D | line 18 | <mark style="background-color: #fff2a8; color: #1f2328;">return self.relu(self.fc(x))</mark>
2E | line 29 | <mark style="background-color: #fff2a8; color: #1f2328;">a_v = self.attention_V(x)  # [B, N, D]</mark>
2E | line 30 | <mark style="background-color: #fff2a8; color: #1f2328;">a_u = self.attention_U(x)</mark>
2G | line 56 | <mark style="background-color: #fff2a8; color: #1f2328;">attention = F.softmax(a_logits, dim=-1)</mark>
2H | line 57 | <mark style="background-color: #fff2a8; color: #1f2328;">bag_feat = torch.bmm(attention, med_feat).squeeze(1)</mark>
2I | line 58 | <mark style="background-color: #fff2a8; color: #1f2328;">logits = self.classifier(bag_feat)    # [B, n_class], raw logits</mark></code></pre>

**Additional completion:** the model also masks padded patches before attention normalization, so they receive zero attention weight.

In [ ]:
class Classifier1FC(nn.Module):
    def __init__(self, n_channels, n_classes, droprate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(droprate) if droprate > 0 else nn.Identity()
        self.fc = nn.Linear(n_channels, n_classes)

    def forward(self, x):
        return self.fc(self.dropout(x))


class DimReduction(nn.Module):
    def __init__(self, n_channels, m_dim):
        super().__init__()
        self.fc = nn.Linear(n_channels, m_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.fc(x))


class AttentionGated(nn.Module):
    def __init__(self, L=256, D=128, K=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_weights_w = nn.Linear(D, K)

    def forward(self, x):
        a_v = self.attention_V(x)  # [B, N, D]
        a_u = self.attention_U(x)
        return self.attention_weights_w(a_v * a_u).transpose(1, 2)


class ABMIL(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.dimreduction = DimReduction(conf.D_feat, conf.D_inner)
        self.attention = AttentionGated(conf.D_inner, conf.D_attn, K=1)
        self.classifier = Classifier1FC(conf.D_inner, conf.n_class, conf.droprate)

    def forward(self, x, return_attention=False, mask=None):
        # The optional mask also preserves the supplied WSI visualizer's API.
        if x.ndim != 3 or x.shape[1] == 0:
            raise ValueError('Expected nonempty input [B, N, D_feat].')
        if mask is not None:
            if mask.shape != x.shape[:2] or mask.dtype != torch.bool:
                raise ValueError('mask must be boolean with shape [B, N].')
            mask = mask.to(x.device)
            if not mask.any(dim=1).all():
                raise ValueError('Every slide must contain at least one real patch.')
            x = x.masked_fill(~mask.unsqueeze(-1), 0)
        med_feat = self.dimreduction(x)       # [B, N, D_inner]
        a_logits = self.attention(med_feat)   # [B, 1, N]
        if mask is not None:
            a_logits = a_logits.masked_fill(~mask.unsqueeze(1), float('-inf'))
        attention = F.softmax(a_logits, dim=-1)
        bag_feat = torch.bmm(attention, med_feat).squeeze(1)
        logits = self.classifier(bag_feat)    # [B, n_class], raw logits
        return (logits, attention) if return_attention else logits

**Sanity Check**

Before running the full training pipeline, check the tensor shapes and attention normalization.


In [ ]:
model_cpu = ABMIL(conf).eval()
x_demo = torch.randn(2, 7, conf.D_feat)
mask_demo = torch.tensor([[True] * 7, [True] * 4 + [False] * 3])
with torch.no_grad():
    logits_demo, attn_demo = model_cpu(x_demo, return_attention=True, mask=mask_demo)
    single_logits = model_cpu(x_demo[1:2, :4])
    changed_padding = x_demo.clone()
    changed_padding[1, 4:] = 1e6
    changed_logits = model_cpu(changed_padding, mask=mask_demo)
    permutation = torch.randperm(7)
    permuted_logits = model_cpu(x_demo[:, permutation], mask=mask_demo[:, permutation])
print('logits:', logits_demo.shape, '| attention:', attn_demo.shape)
print('attention sums:', attn_demo.sum(dim=-1))
assert logits_demo.shape == (2, int(conf.n_class))
assert attn_demo.shape == (2, 1, 7)
assert torch.count_nonzero(attn_demo[1, :, 4:]) == 0
torch.testing.assert_close(attn_demo.sum(dim=-1), torch.ones(2, 1))
torch.testing.assert_close(logits_demo[1:2], single_logits)
torch.testing.assert_close(logits_demo, changed_logits)
torch.testing.assert_close(logits_demo, permuted_logits)
print('Shapes, normalization, padding invariance, and permutation invariance passed.')

<a id="completed-answers-model"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed written answers</mark> — Model and attention answers

- <mark style="background-color: #fff2a8; color: #1f2328;">Trainable tensors are the weights and biases in the projection, both attention branches, the attention scoring layer, and the final classifier. ReLU, tanh, sigmoid, softmax, and dropout have no learned parameters.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">Softmax attention is nonnegative and sums to one over real patches in each slide. Masked positions receive exactly zero weight. The weighted sum is permutation invariant, while reordering patches reorders their attention weights.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">Other nonnegative normalizations or sparse attention functions are possible. Independent sigmoid scores need normalization if a convex weighted average is desired; otherwise bag size changes the representation scale. Changing the normalization is a model ablation, so it requires validation.</mark>

# Step 3 - Device, loss, and optimizer

For multi-class slide classification, use cross-entropy as the baseline objective and AdamW as the baseline optimizer. Complete the code below.

[nn.CrossEntropyLoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html)

[torch.optim.AdamW](https://docs.pytorch.org/docs/2.14/generated/torch.optim.AdamW.html)

<a id="completed-code-step-3"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Step 3: Loss and optimizer

Filled statements in the executable cell below:

<pre><code>3A | line 06 | <mark style="background-color: #fff2a8; color: #1f2328;">criterion = nn.CrossEntropyLoss()</mark>
3B | line 07 | <mark style="background-color: #fff2a8; color: #1f2328;">optimizer = torch.optim.AdamW(model.parameters(), lr=float(conf.lr), weight_decay=float(conf.wd))</mark></code></pre>

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Reset after all exploratory cells so initialization and minibatch order match ablations.
seed_everything(int(conf.seed))
train_loader.generator.manual_seed(int(conf.seed))
model = ABMIL(conf).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=float(conf.lr), weight_decay=float(conf.wd))
print('Device:', device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

# Step 4 - Complete one training epoch

The essential PyTorch optimization sequence is:

`model.train()` -> move tensors to device -> `zero_grad()` -> forward pass -> compute loss -> `backward()` -> `step()`.

Implement this sequence yourself. This is one of the main learning objectives of the project.

[Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)

[train() vs eval()](https://discuss.pytorch.org/t/model-train-and-model-eval-vs-model-and-model-eval/5744)

<a id="completed-code-step-4"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Step 4: Training epoch

Filled statements in the executable cell below:

<pre><code>4A | line 02 | <mark style="background-color: #fff2a8; color: #1f2328;">model.train()</mark>
4B | line 05 | <mark style="background-color: #fff2a8; color: #1f2328;">bag = data[&#x27;input&#x27;].to(device, dtype=torch.float32, non_blocking=True)</mark>
4B | line 06 | <mark style="background-color: #fff2a8; color: #1f2328;">labels = data[&#x27;label&#x27;].to(device, dtype=torch.long, non_blocking=True)</mark>
4C | line 08 | <mark style="background-color: #fff2a8; color: #1f2328;">optimizer.zero_grad(set_to_none=True)</mark>
4D | line 09 | <mark style="background-color: #fff2a8; color: #1f2328;">logits = model(bag, mask=mask)</mark>
4E | line 10 | <mark style="background-color: #fff2a8; color: #1f2328;">loss = criterion(logits, labels)</mark>
4F | line 13 | <mark style="background-color: #fff2a8; color: #1f2328;">loss.backward()</mark>
4G | line 14 | <mark style="background-color: #fff2a8; color: #1f2328;">optimizer.step()</mark></code></pre>

In [ ]:
def train_one_epoch(model, criterion, data_loader, optimizer, device):
    model.train()
    total_loss, total_correct, total_examples = 0.0, 0, 0
    for data in data_loader:
        bag = data['input'].to(device, dtype=torch.float32, non_blocking=True)
        labels = data['label'].to(device, dtype=torch.long, non_blocking=True)
        mask = data['mask'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(bag, mask=mask)
        loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite training loss.')
        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size
    if total_examples == 0:
        raise ValueError('Training loader is empty.')
    return {'loss': total_loss / total_examples, 'acc': 100.0 * total_correct / total_examples}

# Step 5 - Validation and test evaluation

Evaluation should not update the model. Complete the missing lines and use the provided metric helper for macro-AUROC and macro-F1.


<a id="completed-code-step-5"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Step 5: Evaluation

Filled statements in the executable cell below:

<pre><code>5A | line 18 | <mark style="background-color: #fff2a8; color: #1f2328;">model.eval()</mark>
5B | line 22 | <mark style="background-color: #fff2a8; color: #1f2328;">bag = data[&#x27;input&#x27;].to(device, dtype=torch.float32, non_blocking=True)</mark>
5B | line 23 | <mark style="background-color: #fff2a8; color: #1f2328;">labels = data[&#x27;label&#x27;].to(device, dtype=torch.long, non_blocking=True)</mark>
5C | line 25 | <mark style="background-color: #fff2a8; color: #1f2328;">logits = model(bag, mask=mask)</mark>
5C | line 26 | <mark style="background-color: #fff2a8; color: #1f2328;">loss = criterion(logits, labels)</mark>
5D | line 27 | <mark style="background-color: #fff2a8; color: #1f2328;">probabilities = F.softmax(logits, dim=1)</mark></code></pre>

In [ ]:
def calibration_metrics(probabilities, labels, n_bins=15):
    # Top-label ECE: fixed equal-width bins; Brier: sum over classes, mean over slides.
    confidence, prediction = probabilities.max(dim=1)
    correct = prediction.eq(labels).float()
    bin_index = (confidence * n_bins).long().clamp(max=n_bins - 1)
    ece = torch.zeros((), dtype=probabilities.dtype)
    for i in range(n_bins):
        selected = bin_index == i
        if selected.any():
            ece += selected.float().mean() * (correct[selected].mean() - confidence[selected].mean()).abs()
    targets = F.one_hot(labels, num_classes=probabilities.shape[1]).float()
    brier = (probabilities - targets).square().sum(dim=1).mean()
    return {'ece': float(ece), 'brier': float(brier)}


@torch.no_grad()
def evaluate(model, criterion, data_loader, device, conf, return_predictions=False):
    model.eval()
    total_loss, total_examples = 0.0, 0
    probabilities_all, labels_all, slide_ids = [], [], []
    for data in data_loader:
        bag = data['input'].to(device, dtype=torch.float32, non_blocking=True)
        labels = data['label'].to(device, dtype=torch.long, non_blocking=True)
        mask = data['mask'].to(device, non_blocking=True)
        logits = model(bag, mask=mask)
        loss = criterion(logits, labels)
        probabilities = F.softmax(logits, dim=1)
        total_loss += loss.item() * labels.shape[0]
        total_examples += labels.shape[0]
        probabilities_all.append(probabilities.cpu())
        labels_all.append(labels.cpu())
        slide_ids.extend(data['slide_id'])
    if total_examples == 0:
        raise ValueError('Evaluation loader is empty.')
    probabilities_all = torch.cat(probabilities_all)
    labels_all = torch.cat(labels_all)
    metrics = classification_metrics(probabilities_all, labels_all, conf.n_class)
    metrics['loss'] = total_loss / total_examples
    metrics.update(calibration_metrics(probabilities_all, labels_all))
    if return_predictions:
        predictions = pd.DataFrame({'slide_id': slide_ids, 'label': labels_all.numpy(),
                                    'prediction': probabilities_all.argmax(1).numpy()})
        for c in range(int(conf.n_class)):
            predictions[f'p_{c}'] = probabilities_all[:, c].numpy()
        return metrics, predictions
    return metrics

<a id="completed-answers-training"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed written answers</mark> — Training and evaluation answers

- <mark style="background-color: #fff2a8; color: #1f2328;"><code>model.eval()</code> changes modules such as dropout and batch normalization to evaluation behavior. <code>torch.no_grad()</code> disables autograd recording, saving memory and computation. Evaluation uses both; neither is a substitute for the other.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">PyTorch adds newly computed gradients to existing <code>.grad</code> buffers. Without <code>zero_grad()</code>, each update would include previous batches&#x27; gradients, unintentionally changing the optimization algorithm. Deliberate gradient accumulation instead clears gradients at controlled intervals and scales the loss.</mark>
- <mark style="background-color: #fff2a8; color: #1f2328;">Selecting an epoch using test performance leaks held-out information into model selection and makes the reported test estimate optimistic. Here every checkpoint is selected by validation macro-AUROC + macro-F1; the test set is evaluated only after that selection is complete.</mark>

# Step 6 - Train, validate, and select the best checkpoint

Use the validation set to choose the best epoch. The test set must remain untouched until model selection is complete.

For debugging, set `NUM_EPOCHS = 2`. For your reported baseline, use the full configured epoch count unless instructed otherwise.


In [ ]:
NUM_EPOCHS = int(conf.train_epoch)  # Full baseline: 30 epochs. Use 2 only for debugging.
REUSE_COMPLETED_RUNS = True

def experiment_signature(conf, datasets, num_epochs, smoothing):
    # Changes to code, config, split membership, or the feature file invalidate cached results.
    notebook = json.loads((PROJECT_DIR / 'ECE1512_ProjectA_Student.ipynb').read_text(encoding='utf-8'))
    source = '\n'.join(''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type'] == 'code')
    feature = Path(conf.data_dir)
    details = dict(config=vars(conf), epochs=num_epochs, smoothing=smoothing,
                   splits=[list(ds.data_names) for ds in datasets],
                   feature_size=feature.stat().st_size, feature_mtime=feature.stat().st_mtime_ns,
                   code_sha256=hashlib.sha256(source.encode()).hexdigest(),
                   torch_version=str(torch.__version__), device=str(device))
    return hashlib.sha256(json.dumps(details, sort_keys=True).encode()).hexdigest()

def fit_model(model, train_criterion, optimizer, train_loader, val_loader,
              conf, num_epochs, run_dir, signature):
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / 'best.pt'
    if REUSE_COMPLETED_RUNS and checkpoint_path.exists():
        saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        if saved['signature'] == signature and saved['epochs_completed'] == num_epochs:
            print(f'Reusing completed {run_dir.name}: {num_epochs} epochs, best epoch {saved["best_epoch"] + 1}')
            model.load_state_dict(saved['model_state'])
            return saved['history'], saved['best_epoch'], saved['model_state']

    best_score, best_epoch, best_state, history = float('-inf'), -1, None, []
    # Hard-target CE on validation/test gives comparable NLL for both objectives.
    eval_criterion = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        start = time.time()
        train_metrics = train_one_epoch(model, train_criterion, train_loader, optimizer, device)
        val_metrics = evaluate(model, eval_criterion, val_loader, device, conf)
        score = val_metrics['auc'] + val_metrics['f1']
        if not np.isfinite(score):
            raise FloatingPointError('Validation selection score is not finite.')
        if score > best_score:  # A tie keeps the earliest checkpoint.
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        row = dict(epoch=epoch, train_loss=train_metrics['loss'], train_acc=train_metrics['acc'],
                   **{f'val_{k}': v for k, v in val_metrics.items()})
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
        print(f'{run_dir.name} | Epoch {epoch+1:02d}/{num_epochs} | '
              f'train loss={row["train_loss"]:.4f} acc={row["train_acc"]:.2f} | '
              f'val loss={row["val_loss"]:.4f} auc={row["val_auc"]:.3f} f1={row["val_f1"]:.3f} | '
              f'best={best_epoch+1} | {time.time()-start:.1f}s', flush=True)
    if best_state is None:
        raise ValueError('At least one training epoch is required.')
    torch.save(dict(model_state=best_state, best_epoch=best_epoch, best_score=best_score,
                    epochs_completed=num_epochs, history=history, config=vars(conf), signature=signature),
               checkpoint_path)
    return history, best_epoch, best_state

baseline_dir = OUTPUT_DIR / f'{DATASET}_baseline'
baseline_signature = experiment_signature(conf, (train_data, val_data, test_data), NUM_EPOCHS, 0.0)
history, best_epoch, best_model_state = fit_model(
    model, criterion, optimizer, train_loader, val_loader, conf, NUM_EPOCHS,
    baseline_dir, baseline_signature)

In [ ]:
# Plot learning curves.
epochs = [x['epoch'] + 1 for x in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs, [x['train_loss'] for x in history], label='train loss')
plt.plot(epochs, [x['val_loss'] for x in history], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# Step 7 - Final held-out test evaluation

Only now, after the best checkpoint has been chosen from validation performance, load it and evaluate once on the test set.


In [ ]:
def final_test(model, state, test_loader, conf, run_dir, signature):
    model.load_state_dict(state)
    metrics_path = run_dir / 'test_metrics.json'
    predictions_path = run_dir / 'test_predictions.csv'
    if REUSE_COMPLETED_RUNS and metrics_path.exists() and predictions_path.exists():
        saved = json.loads(metrics_path.read_text())
        if saved['signature'] == signature:
            print('Loaded previously recorded final test:', run_dir.name)
            return saved['metrics'], pd.read_csv(predictions_path)
    metrics, predictions = evaluate(model, nn.CrossEntropyLoss(), test_loader, device, conf,
                                    return_predictions=True)
    predictions.to_csv(predictions_path, index=False)
    metrics_path.write_text(json.dumps({'signature': signature, 'metrics': metrics}, indent=2))
    return metrics, predictions

test_metrics, test_predictions = final_test(
    model, best_model_state, test_loader, conf, baseline_dir, baseline_signature)
print('Best validation epoch:', best_epoch + 1)
print('Final test metrics (acc in %, all other metrics unscaled):')
for key, value in test_metrics.items():
    print(f'  {key}: {value:.4f}')

# Step 8 -  Inspect Attention on One Slide

Attention is useful for understanding which instances contribute strongly to the learned bag representation, but attention weights should not automatically be interpreted as causal explanations.


In [ ]:
one_batch = mil_collate_fn([test_data[0]])
model.eval()
with torch.no_grad():
    logits, attention = model(one_batch['input'].to(device, dtype=torch.float32),
                              return_attention=True, mask=one_batch['mask'].to(device))
print('Slide:', one_batch['slide_id'][0])
print('Predicted class:', int(logits.argmax(dim=1)[0]))
print('Attention shape:', tuple(attention.shape))
print('Attention sum for first slide:', float(attention[0].sum()))

## Visualize Attention on Whole-Slide Images (WSIs)

Before running the attention visualization, please complete the following steps:

1. Visit the [ECE1512-WSI-Feature dataset on Hugging Face](https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature)

2. Download the original WSIs `test_067.tif` and `test_068.tif` and the corresponding tumor annotation file `test_068.xml`.

3. Create a folder named `slide/` under your project directory and place the downloaded files in this folder.

Run this cell after training and final test evaluation, with `DATASET = 'C16'`. It restores `best_model_state` and plots the two slides in `PROJECT_DIR/slide`:

- **test_068 (tumor):** attention overlay and green contours from `test_068.xml`, including a zoom of the annotated region.
- **test_067 (normal):** attention overlay without annotation contours, including a zoom around the highest-attention patch.

The blue rectangle marks the zoom shown on the right. Colors show **attention percentiles within each slide**, not patch tumor probabilities; areas without extracted patches stay transparent. The displayed `P(tumor)` is the model's slide-level prediction.

HDF5 patch coordinates and XML contour coordinates are assumed to be expressed in level-0 `(x, y)`  coordinates; patch coordinates are treated as top-left corners. The function parameter `patch_size_level0=512` is an editable **assumption** for the patch footprint because these HDF5 files do not store patch-size metadata. Match it to the feature extractor's patch size for an exact footprint. All visualization functions are in `wsi_visualization.py`; the cell below only calls `visualize_wsi_attention`.

[OpenSlide Python](https://openslide.org/api/python/) reads a thumbnail and a pyramid-level crop rather than loading the full-resolution WSI into memory. If dependencies are missing, run `%pip install openslide-python openslide-bin` and restart the kernel if needed. Raw per-patch attention and coordinates are retained in `attention_visualizations`.


In [ ]:
attention_visualizations = {}
if DATASET == 'C16':
    needed = [PROJECT_DIR / 'slide' / name for name in ('test_067.tif', 'test_068.tif', 'test_068.xml')]
    absent = [str(p) for p in needed if not p.is_file()]
    if absent:
        print('WSI overlays are unavailable until these raw assets are downloaded:', absent)
        print('Run scripts/download_slides.py, then rerun this cell. HDF5-only training is complete independently.')
    else:
        attention_visualizations = visualize_wsi_attention(model, conf, PROJECT_DIR, best_model_state)
        for slide_id, values in attention_visualizations.items():
            np.savez_compressed(baseline_dir / f'{slide_id}_attention.npz', **values)

# Step 9 - Research extension: propose two ideas, implement one of them

After completing the ABMIL baseline, propose **two distinct research ideas**. Select **one** for implementation and evaluation on **CAMELYON16 (C16), CAMELYON17 (C17), and BRACS**. You may explore explainability, more advanced model structures, improved objectives beyond cross-entropy, or another well-motivated direction.

**A negative result will not be penalized if the design choices, experiments, observations, and possible reasons for failure are well justified.** Report evidence relevant to your hypothesis: an explainability idea, for example, needs an explanation evaluation in addition to classification results.

Feel free to contact your TA to discuss your research ideas.

<a id="completed-research-answers"></a>

## <mark style="background-color: #fff2a8; color: #1f2328;">Completed written answers</mark> — Two proposed research ideas

<strong><mark style="background-color: #fff2a8; color: #1f2328;">Idea 1 (selected): label smoothing for slide-level confidence.</mark></strong> Weak slide labels and limited slide counts can produce overly confident classifiers. Replace the one-hot target with
$q_c=(1-\epsilon)\mathbf{1}[c=y]+\epsilon/C$, with a fixed $\epsilon=0.1$, and minimize $-\sum_c q_c\log p_c$.
The hypothesis is lower held-out negative log likelihood (NLL), Brier score, and expected calibration error (ECE), with competitive macro-AUROC and macro-F1. This tests an established regularizer in this MIL setting; it is not a claim of a new algorithm. Excessive smoothing may cause underconfidence or suppress rare-class evidence, so improvement is not assumed.

<strong><mark style="background-color: #fff2a8; color: #1f2328;">Idea 2 (proposed, not implemented): diversity-regularized two-head attention.</mark></strong> Learn two normalized attention distributions, concatenate their pooled representations, and discourage identical attention maps using a cosine-overlap penalty. The hypothesis is that distinct heads capture complementary tissue regions and reduce dependence on one region. Evaluate classification together with prediction changes after removing each head's top-ranked patches, compared with random removal; compare against a parameter-matched two-head model without the penalty. Potential failures include selecting irrelevant tissue or discouraging useful overlap. This idea would require a separate validation-only regularization search.

<strong><mark style="background-color: #fff2a8; color: #1f2328;">Protocol for the selected idea.</mark></strong> Run baseline CE and smoothed CE for all three datasets, each for the configured 30 epochs, using full bags, full training splits, seed 42, batch size 1, the same gated ABMIL architecture, AdamW, learning rate, weight decay, and initial weights/minibatch order. Select the epoch only by validation macro-AUROC + macro-F1, breaking ties in favor of the earlier epoch. Test each selected checkpoint once and retain its predictions. Validation and test always use ordinary CE so the reported NLL is comparable. The three-class settings are binary C16, four-class C17, and seven-class BRACS; BRACS labels are not regrouped.

Report accuracy (%), macro-AUROC, macro-F1, hard-target NLL, multiclass Brier score, and top-label ECE with 15 fixed equal-width confidence bins. Lower NLL/Brier/ECE is better. ECE is a finite-sample binning estimate, so reliability plots and the other metrics are considered together. No temperature fitting or test-based hyperparameter tuning is performed. Checkpoints, split lists, histories, configurations, environment versions, and per-slide test probabilities are saved under `results/`.

<strong><mark style="background-color: #fff2a8; color: #1f2328;">Limitations fixed in advance.</mark></strong> One seed cannot establish statistical significance. The supplied C17 train/validation split is at slide level and can share patients; its held-out test centers remain disjoint. We retain the provided split for the required comparison and report patient overlap below. Raw feature extraction and WSI patch-size metadata are outside this experiment.

References: [Ilse et al., Attention-based Deep Multiple Instance Learning (2018)](https://proceedings.mlr.press/v80/ilse18a.html); [Müller et al., When Does Label Smoothing Help? (2019)](https://arxiv.org/abs/1906.02629).

<a id="completed-extension-settings"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Research extension: experiment settings

The following cell defines the fixed label-smoothing strength and the two comparison methods.

In [ ]:
# Fixed before examining test results; epsilon is not tuned on held-out slides.
LABEL_SMOOTHING = 0.1
RUN_ALL_DATASETS = True
METHODS = {'baseline': 0.0, 'label_smoothing': LABEL_SMOOTHING}

<a id="completed-extension-experiments"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Research extension: implementation and experiments

The following cell implements the split audit, baseline/extension runs for all three datasets, validation-selected testing, and saved comparison results.

In [ ]:
def audit_splits(datasets, conf, run_dir):
    names = ('train', 'validation', 'test')
    groups = [set(ds.data_names) for ds in datasets]
    assert all(groups), 'All splits must contain slides.'
    assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
    import h5py
    from dataloader import _map_label
    rows = []
    with h5py.File(conf.data_dir, 'r') as h5:
        for split, dataset in zip(names, datasets):
            for slide_id in dataset.data_names:
                label = _map_label(int(h5[slide_id].attrs['label']), conf.dataset, conf.n_class)
                assert 0 <= label < int(conf.n_class)
                rows.append(dict(split=split, slide_id=slide_id, label=label,
                                 patches=int(h5[slide_id]['feat'].shape[0])))
    splits = pd.DataFrame(rows)
    run_dir.mkdir(parents=True, exist_ok=True)
    splits.to_csv(run_dir / 'splits.csv', index=False)
    print(conf.dataset, 'split/class counts:')
    display(pd.crosstab(splits['split'], splits['label']).reindex(names))
    if conf.dataset == 'CAMELYON17':
        metadata = pd.read_csv(Path(conf.dataset_csv_dir) / 'camelyon17.csv').set_index('slide_id')
        patients = [set(metadata.loc[list(group), 'case_id']) for group in groups]
        centers = [set(metadata.loc[list(group), 'center']) for group in groups]
        print('C17 shared train/validation patients:', len(patients[0] & patients[1]))
        print('C17 centers (train, validation, test):', centers)
        assert not ((patients[0] | patients[1]) & patients[2])
    return splits

def make_result(dataset, method, epochs, best_epoch, metrics, run_dir):
    return dict(dataset=dataset, method=method, epochs=epochs, best_epoch=best_epoch + 1,
                **metrics, run_dir=str(run_dir.relative_to(PROJECT_DIR)))

experiment_results = []
datasets_to_run = list(CONFIG_FILES) if RUN_ALL_DATASETS else [DATASET]
for dataset_name in datasets_to_run:
    run_conf = load_config(PROJECT_DIR / 'config' / CONFIG_FILES[dataset_name])
    seed_everything(int(run_conf.seed))
    datasets = build_HDF5_feat_dataset(run_conf.data_dir, run_conf)
    run_conf.D_feat = infer_feature_dim(datasets[0])
    split_table = audit_splits(datasets, run_conf, OUTPUT_DIR / dataset_name)
    for method, smoothing in METHODS.items():
        run_dir = OUTPUT_DIR / f'{dataset_name}_{method}'
        num_epochs = int(run_conf.train_epoch)
        signature = experiment_signature(run_conf, datasets, num_epochs, smoothing)
        # Reuse the baseline already trained in Steps 1-7 (only if it was a full run).
        if dataset_name == DATASET and method == 'baseline' and signature == baseline_signature:
            run_metrics, run_best_epoch = test_metrics, best_epoch
        else:
            seed_everything(int(run_conf.seed))
            loaders = make_loaders(*datasets, run_conf)
            run_model = ABMIL(run_conf).to(device)
            run_optimizer = torch.optim.AdamW(run_model.parameters(), lr=float(run_conf.lr),
                                              weight_decay=float(run_conf.wd))
            run_criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)
            run_history, run_best_epoch, run_state = fit_model(
                run_model, run_criterion, run_optimizer, loaders[0], loaders[1], run_conf,
                num_epochs, run_dir, signature)
            run_metrics, _ = final_test(run_model, run_state, loaders[2], run_conf, run_dir, signature)
            del run_model, run_optimizer, run_state
        experiment_results.append(make_result(dataset_name, method, num_epochs, run_best_epoch,
                                              run_metrics, run_dir))
        pd.DataFrame(experiment_results).to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
        print(f'{dataset_name} {method}: {run_metrics}', flush=True)

results_df = pd.DataFrame(experiment_results)
display(results_df.drop(columns='run_dir').round(4))
import importlib.metadata
environment = {name: importlib.metadata.version(name)
               for name in ['torch', 'torchmetrics', 'h5py', 'numpy', 'pandas', 'scikit-learn']}
environment.update(python=sys.version, device=str(device),
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2))

<a id="completed-extension-analysis"></a>

### <mark style="background-color: #fff2a8; color: #1f2328;">Completed code</mark> — Research extension: plots and written observations

The following cell produces the learning/calibration plots, metric differences, and written observations from the measured results.

In [ ]:
# Compare only saved held-out predictions; these plots do not trigger test re-evaluation.
fig, axes = plt.subplots(2, len(datasets_to_run), figsize=(5 * len(datasets_to_run), 8), squeeze=False)
for col, dataset_name in enumerate(datasets_to_run):
    ax_loss, ax_calibration = axes[:, col]
    ax_calibration.plot([0, 1], [0, 1], '--', color='gray', label='ideal')
    for method in METHODS:
        run_dir = OUTPUT_DIR / f'{dataset_name}_{method}'
        run_history = pd.read_csv(run_dir / 'history.csv')
        ax_loss.plot(run_history.epoch + 1, run_history.val_loss, label=method)
        predictions = pd.read_csv(run_dir / 'test_predictions.csv')
        p = predictions.filter(regex=r'^p_\d+$').to_numpy()
        confidence, predicted = p.max(1), p.argmax(1)
        bin_index = np.minimum((confidence * 15).astype(int), 14)
        xs, ys, counts = [], [], []
        for i in range(15):
            selected = bin_index == i
            if selected.any():
                xs.append(confidence[selected].mean())
                ys.append((predicted[selected] == predictions.label.to_numpy()[selected]).mean())
                counts.append(int(selected.sum()))
        ax_calibration.plot(xs, ys, marker='o', label=method)
        pd.DataFrame({'mean_confidence': xs, 'accuracy': ys, 'count': counts}).to_csv(
            run_dir / 'reliability_bins.csv', index=False)
    ax_loss.set(title=f'{dataset_name}: validation NLL', xlabel='Epoch', ylabel='Hard-target CE')
    ax_calibration.set(title=f'{dataset_name}: test reliability', xlabel='Mean confidence',
                       ylabel='Empirical accuracy', xlim=(0, 1), ylim=(0, 1))
    for ax in (ax_loss, ax_calibration):
        ax.legend()
        ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'learning_and_calibration.png', dpi=160, bbox_inches='tight')
plt.show()

delta_rows = []
for dataset_name in datasets_to_run:
    subset = results_df.set_index(['dataset', 'method'])
    baseline = subset.loc[(dataset_name, 'baseline')]
    extension = subset.loc[(dataset_name, 'label_smoothing')]
    delta_rows.append(dict(dataset=dataset_name, **{
        metric: extension[metric] - baseline[metric]
        for metric in ['auc', 'f1', 'acc', 'loss', 'brier', 'ece']}))
deltas = pd.DataFrame(delta_rows).set_index('dataset')
print('Label smoothing minus baseline (negative loss/Brier/ECE is better):')
display(deltas.round(4))
deltas.to_csv(OUTPUT_DIR / 'extension_deltas.csv')

observations = ['### Observations from this run']
for dataset_name, row in deltas.iterrows():
    improved = [name for name in ['loss', 'brier', 'ece'] if row[name] < 0]
    observations.append(
        f'- **{dataset_name}:** macro-AUROC changed by {row.auc:+.4f}, macro-F1 by {row.f1:+.4f}, '
        f'and accuracy by {row.acc:+.2f} percentage points. '
        f'Lower calibration/probability metrics: {", ".join(improved) if improved else "none"}. '
        f'NLL delta {row.loss:+.4f}; Brier delta {row.brier:+.4f}; ECE delta {row.ece:+.4f}.')
observations.append(
    'Interpret these as paired observations at seed 42, not statistical evidence across seeds. '
    'Smoothing can reduce overconfidence, but it can also cause underconfidence; changes in the '
    'validation-selected epoch can contribute to test differences. Dataset class count, imbalance, '
    'center shift, and small validation sets can explain inconsistent effects. These are possible '
    'mechanisms, not causal findings. Check NLL/Brier together with the reliability curves and macro-F1. '
    'Repeated seeds and a patient-grouped C17 validation split are useful follow-up experiments; '
    'the reported comparison preserves the supplied protocol.')
display(Markdown('\n\n'.join(observations)))